## Overview


text preprcessing steps carried out:
1. Lower casing 
2. Remover of punctuation
3. Removal of stopwords
4. Removal of frequent words
5. Lemmatization

Note: I used **lemmatization** over **stemming** because it preserve the ***root*** word, even though it more slower during preprocessing

In [129]:
import numpy as np
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import string
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

In [130]:
df = pd.read_csv("fake_or_real_news.csv")

In [131]:
df.head()

,Unnamed: 0,title,text,label
0,8476,You Can Smell Hillary’s Fear,"Daniel Greenfield, a Shillman Journalism Fello...",FAKE
1,10294,Watch The Exact Moment Paul Ryan Committed Pol...,Google Pinterest Digg Linkedin Reddit Stumbleu...,FAKE
2,3608,Kerry to go to Paris in gesture of sympathy,U.S. Secretary of State John F. Kerry said Mon...,REAL
3,10142,Bernie supporters on Twitter erupt in anger ag...,"— Kaydee King (@KaydeeKing) November 9, 2016 T...",FAKE
4,875,The Battle of New York: Why This Primary Matters,It's primary day in New York and front-runners...,REAL


In [132]:
df.drop(columns ="Unnamed: 0", inplace = True)

In [133]:
print(f"\n{df.head()}")
print("="* 80)
print(df.info())
print("="* 80)
print(f"Total missing values : {df.isnull().isnull().sum()}")
print("-"* 80)
print(f"Number of duplicated clumns = {df.duplicated().sum()}")


                                               title  \
0                       You Can Smell Hillary’s Fear   
1  Watch The Exact Moment Paul Ryan Committed Pol...   
2        Kerry to go to Paris in gesture of sympathy   
3  Bernie supporters on Twitter erupt in anger ag...   
4   The Battle of New York: Why This Primary Matters   

                                                text label  
0  Daniel Greenfield, a Shillman Journalism Fello...  FAKE  
1  Google Pinterest Digg Linkedin Reddit Stumbleu...  FAKE  
2  U.S. Secretary of State John F. Kerry said Mon...  REAL  
3  — Kaydee King (@KaydeeKing) November 9, 2016 T...  FAKE  
4  It's primary day in New York and front-runners...  REAL  
<class 'pandas.DataFrame'>
RangeIndex: 6335 entries, 0 to 6334
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   title   6335 non-null   str  
 1   text    6335 non-null   str  
 2   label   6335 non-null   str  
dtypes: str(3)
memory usag

# Lower case
Convert the case to lower case for uniformity across text with diferent case

In [134]:
df["prep_title"], df["prep_text"] = df["title"].str.lower(), df["text"].str.lower()

In [135]:
df.head()

,title,text,label,prep_title,prep_text
0,You Can Smell Hillary’s Fear,"Daniel Greenfield, a Shillman Journalism Fello...",FAKE,you can smell hillary’s fear,"daniel greenfield, a shillman journalism fello..."
1,Watch The Exact Moment Paul Ryan Committed Pol...,Google Pinterest Digg Linkedin Reddit Stumbleu...,FAKE,watch the exact moment paul ryan committed pol...,google pinterest digg linkedin reddit stumbleu...
2,Kerry to go to Paris in gesture of sympathy,U.S. Secretary of State John F. Kerry said Mon...,REAL,kerry to go to paris in gesture of sympathy,u.s. secretary of state john f. kerry said mon...
3,Bernie supporters on Twitter erupt in anger ag...,"— Kaydee King (@KaydeeKing) November 9, 2016 T...",FAKE,bernie supporters on twitter erupt in anger ag...,"— kaydee king (@kaydeeking) november 9, 2016 t..."
4,The Battle of New York: Why This Primary Matters,It's primary day in New York and front-runners...,REAL,the battle of new york: why this primary matters,it's primary day in new york and front-runners...


# Remove punctuation
Remove the punctuations from the title and text columns 

In [136]:
df.drop(columns = ["title", "text"], inplace = True)

In [137]:
PUNC_TO_REMOVE = string.punctuation
df["title"] = df["prep_title"].apply(lambda title: title.translate(str.maketrans("","", PUNC_TO_REMOVE)))
df["text"] = df["prep_text"].apply(lambda text: text.translate(str.maketrans("", "", PUNC_TO_REMOVE)))

In [138]:
df.drop(columns = ["prep_title", "prep_text"], inplace = True)

In [139]:
df.head()

,label,title,text
0,FAKE,you can smell hillary’s fear,daniel greenfield a shillman journalism fellow...
1,FAKE,watch the exact moment paul ryan committed pol...,google pinterest digg linkedin reddit stumbleu...
2,REAL,kerry to go to paris in gesture of sympathy,us secretary of state john f kerry said monday...
3,FAKE,bernie supporters on twitter erupt in anger ag...,— kaydee king kaydeeking november 9 2016 the l...
4,REAL,the battle of new york why this primary matters,its primary day in new york and frontrunners h...


# Remova of stopwords

In [157]:

STOPWORDS = set(stopwords.words("english"))
def remove_stopwords(text):
    """custom function to remove the stopwords"""
    return " ".join(word for word in str(text).split() if word not in STOPWORDS)

df["title"] = df["title"].apply(lambda text: remove_stopwords(text))
df["text"] = df["text"].apply(lambda text: remove_stopwords(text))

df.head(2)

,label,title,text
0,FAKE,smell hillary’s fear,daniel greenfield shillman journalism fellow f...
1,FAKE,watch exact moment paul ryan commit political ...,google pinterest digg linkedin reddit stumbleu...


# Frequent words removing

In [158]:
from collections import Counter
# coutn of frequent words in text column
count = Counter()
# let's use the text_wo_stop column in the dataframe
for text in df["text"]:
    for word in text.split():
        count[word] += 1

count.most_common(10)

[('say', 30411),
 ('trump', 19026),
 ('clinton', 15615),
 ('state', 14775),
 ('would', 12550),
 ('u', 12362),
 ('one', 11439),
 ('people', 11145),
 ('make', 10095),
 ('go', 9784)]

In [159]:
count = Counter()
# coutn of frequent words in title column
for text in df["title"]:
    for word in text.split():
        count[word] += 1

count.most_common(10)

[('trump', 983),
 ('clinton', 672),
 ('hillary', 527),
 ('u', 326),
 ('obama', 281),
 ('new', 277),
 ('donald', 265),
 ('election', 256),
 ('gop', 243),
 ('say', 232)]

In [143]:
# FREQWORDS = set([w for (w, wc) in count.most_common(10)])
# def remove_freqwords(text):
#     """custom function to remove the frequent words"""
#     return " ".join([word for word in str(text).split() if word not in FREQWORDS])

# df["title"] = df["title"].apply(lambda text: remove_freqwords(text))
# df["text"] = df["text"].apply(lambda text: remove_freqwords(text))
# df.head()

# Lemmtization

In [160]:
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\USER\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\USER\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\USER\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [161]:
# from nltk.stem import WordNetLemmatizer

# lemmatizer = WordNetLemmatizer()
# wordnet_map = {"N":wordnet.NOUN, "V":wordnet.VERB, "J":wordnet.ADJ, "R":wordnet.ADV}

# def lemmatize_words(text):
#     return " ".join([lemmatizer.lemmatize(word, "v") for word in text.split()])
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()
wordnet_map = {"N":wordnet.NOUN, "V":wordnet.VERB, "J":wordnet.ADJ, "R":wordnet.ADV}
def lemmatize_words(text):
    pos_tagged_text = nltk.pos_tag(text.split())
    return " ".join(lemmatizer.lemmatize(word, wordnet_map.get(pos[0], wordnet.NOUN)) for word, pos in pos_tagged_text)

df["title"] = df["title"].apply(lambda text: lemmatize_words(text))
df["text"] = df["text"].apply(lambda text: lemmatize_words(text))

df.head()

,label,title,text
0,FAKE,smell hillary’s fear,daniel greenfield shillman journalism fellow f...
1,FAKE,watch exact moment paul ryan commit political ...,google pinterest digg linkedin reddit stumbleu...
2,REAL,kerry go paris gesture sympathy,u secretary state john f kerry say monday stop...
3,FAKE,bernie supporter twitter erupt anger dnc try warn,— kaydee king kaydeeking november 9 2016 lesso...
4,REAL,battle new york primary matter,primary day new york frontrunners hillary clin...


In [162]:
# Drop duplicate columns
df.drop_duplicates(inplace = True)

In [167]:
df.duplicated().sum()

np.int64(0)

In [172]:
df["content"] = df["title"].fillna("") + " " + df["text"].fillna("")
df = df[df["content"].str.strip() != ""]
df = df.drop_duplicates(subset="content").reset_index(drop=True)
df["label"] = df["label"].map({"FAKE": 0, "REAL": 1})

## I have create a .py file that complete the end-to-end pipeline from text preprocessing to training the classification model.

### **Note:** I used Logistic regression & multilayer paerceptron model , where the one with the highest accuracy, best precision and recall wil be selected.

access the .py file from **src** folder